In [ ]:
# - Choose the model whose feature selection you want to run locally.
# - Paste an execution ID from your configured training pipeline, or its full ARN.
# - These inputs do nothing until you run the preparation cell.
MODEL_ID = "usecase2"
EXECUTION_ID = ""


# Run feature selection locally

Run setup, run the entrypoint, then copy the output of the final cell into
`mlops-models/src/mlops_models/entrypoints/feature_selection.py`, replacing its
previous local main block. The final cell outputs only the copyable Python code.

The helper reads the actual FeatureSelection job and its raw input, including
cached producer paths. Selection uses **this checkout's code and parameters**.
The execution cell imports the real entrypoint and calls `execute(...)` directly.

If preparation or selection fails, the notebook prints the original traceback
and continues to the export cell. A failed call leaves `result` as `None` and
retains the exception in `setup_error` or `execution_error` for inspection.

In the VS Code block, change `MODEL_ID` and `EXECUTION_ID` at the top to select
another model and run. A full ARN is used as supplied; a short ID is resolved
using this checkout's `notebooks/config.json`. There is no embedded fallback
account or pipeline. Before selection starts, the block shows the running file,
checkout, parameter file, task type and entrypoint path.

Rerun setup after changing either input. Rerun the execution cell after editing
model selection code. Restart the kernel or Python process when switching
checkouts. The notebook and exported block use the same source-resolution logic;
no change to `launch.json` is required.

The setup and export implementation is in [the helper](../source/feature_selection/support.py).


In [ ]:
# - Resolve the selected job and download/reuse its raw input.
# - Prepare the VS Code block before lookup so it is available if setup fails.
# - Print setup errors and allow the final export cell to run.
from pathlib import Path
import runpy
import sys
import traceback

step = None
vscode_code = None
prepared_inputs = None
setup_error = None
try:
    ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "mlops-models/src/mlops_models").is_dir())
    support = runpy.run_path(str(ROOT / "notebooks/source/feature_selection/support.py"))
    vscode_code = support["_vscode_code"](MODEL_ID, EXECUTION_ID)
    step = support["prepare"](ROOT, MODEL_ID, EXECUTION_ID)
    prepared_inputs = (MODEL_ID, EXECUTION_ID)
except BaseException as error:
    setup_error = error
    print("Input preparation failed; the final cell still provides the VS Code block.")
    traceback.print_exc(file=sys.stdout)


In [ ]:
# - Import and call the real feature-selection entrypoint directly.
# - Print the original traceback on failure and restore the environment.
# - Keep this output separate from the final copyable VS Code block.
import importlib
import sys
import traceback

result = None
execution_error = None
if step is None or prepared_inputs != (MODEL_ID, EXECUTION_ID):
    print("Selection not run. Run the preparation cell successfully for these inputs first.")
else:
    try:
        with support["local_environment"](step):
            from mlops_models.entrypoints import feature_selection

            feature_selection = importlib.reload(feature_selection)
            print("Entrypoint:", feature_selection.__file__, flush=True)
            result = feature_selection.execute(
                model_id=step.model_id,
                input_path=step.data_directory,
                output_dir=step.output_directory,
            )
        print("Returned value:", result)
    except BaseException as error:
        execution_error = error
        print("Feature selection failed; copy the VS Code block from the final cell to debug it.")
        traceback.print_exc(file=sys.stdout)


In [ ]:
# - Output only Python code so Copy Cell Output produces a runnable block.
# - Keep the block available after setup or selection errors.
# - Replace the previous local main block in the entrypoint with this output.
if vscode_code is None:
    print("# No VS Code block available. Fix the setup error and rerun preparation.")
else:
    print(vscode_code, end="")
